# Rastreamento de Objetos com a Biblioteca `trackers` da Roboflow

No notebook anterior, usamos o rastreamento "embutido" na `ultralytics`, que já vem com o `ByteTrack` pronto para uso através do método `track()`. Neste notebook, vamos refazer o mesmo tipo de rastreamento usando a biblioteca [`trackers`](https://github.com/roboflow/trackers), da Roboflow, que separa claramente o detector do rastreador, permitindo trocar de algoritmo de rastreamento sem trocar de modelo de detecção.

Também vamos explorar o `MotionEstimator`, um componente que estima o movimento da própria câmera entre os quadros, e que usamos para deixar os rastros de cada objeto visivelmente mais suaves.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install ultralytics
    !pip install rfdetr
    !pip install moviepy==2.2.1
    !pip install trackers
    !pip install supervision
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas já usadas no notebook anterior, vamos usar mais duas, ambas da Roboflow:
* `trackers`: implementa os algoritmos de rastreamento (`SORTTracker`, `ByteTrackTracker`, etc.), de forma independente de qualquer detector, além do `MotionEstimator`.
* `supervision`: fornece a estrutura `Detections`, usada para representar detecções de forma padronizada, e os "anotadores" (`BoxAnnotator`, `LabelAnnotator`, `TraceAnnotator`, ...) que desenham caixas, rótulos e rastros sobre a imagem.

In [ ]:
from ultralytics import YOLO
from rfdetr import RFDETRNano
from trackers import ByteTrackTracker, OCSORTTracker, MotionEstimator, MotionAwareTraceAnnotator

from collections import defaultdict
from pathlib import Path
from moviepy import VideoFileClip

import cv2
import supervision as sv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image, Video

## 1. A Biblioteca `trackers` da Roboflow

No notebook anterior, o rastreamento veio embutido no método `track()` da própria `ultralytics`, que já vinha com o `ByteTrack` (e o `BoT-SORT`) prontos para uso. A `trackers`, da Roboflow, propõe uma abordagem diferente: ela é só o motor de rastreamento, sem nenhum detector embutido. Essa filosofia é chamada de **BYOD** (*Bring Your Own Detector* ou "traga seu próprio detector"): qualquer modelo que produza caixas delimitadoras (YOLO, RF-DETR, Detectron2, um modelo próprio) pode alimentar o rastreador, desde que as detecções estejam no formato `supervision.Detections`.

Isso separa claramente duas responsabilidades: o detector (`ultralytics`, no nosso caso) só encontra objetos em cada quadro; o rastreador (`trackers`) só decide quais caixas pertencem ao mesmo objeto ao longo do tempo. A vantagem é poder trocar de detector sem trocar de rastreador, ou comparar diferentes algoritmos de rastreamento sobre as mesmas detecções.

A biblioteca já implementa vários algoritmos, todos com a mesma interface, um método `update(detections)` que recebe as detecções do quadro atual e devolve as mesmas detecções com um `tracker_id` atribuído:

* `SORTTracker`: o mais simples e rápido. Usa um filtro de Kalman com modelo de velocidade constante e associação por IoU. Sofre mais com oclusões.
* `ByteTrackTracker`: o mesmo algoritmo do notebook anterior, mas na implementação da Roboflow. Também aproveita detecções de baixa confiança na associação, reduzindo trocas de ID.
* Outros, mais recentes: `OCSORTTracker`, `BoTSORTTracker`, `CBIoUTracker`, `McByteTracker`.

Vamos usar o `ByteTrackTracker` para o rastreamento em si e, na seção 5, explorar o `MotionEstimator`, um componente separado que estima o movimento da própria câmera para deixar os rastros de cada objeto muito mais suaves.

## 2. Rastreamento em um Quadro

Antes de processar o vídeo inteiro, vamos entender a API em um único quadro.

### 2.1. Carregando o modelo

O detector continua sendo a versão 26 do YOLO, como no notebook anterior. A diferença vem a seguir: em vez de chamar `track()`, vamos chamar `predict()` normalmente e entregar as detecções para o rastreador da Roboflow.

In [ ]:
# Caso tenha problemas em usar a versão 26, descomente a linha de baixo e comente a outra
#detect_model = YOLO('modelos/yolov8n.pt')
detect_model = YOLO('modelos/yolo26n.pt')

### 2.2. Carregando um quadro do vídeo

Vamos usar o mesmo vídeo do notebook anterior.

In [ ]:
video_path = 'imagens/05/2099406-hd_1920_1080_30fps.mp4'

cap = cv2.VideoCapture(video_path)
ok, quadro = cap.read()
cap.release()

quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(quadro_rgb)
plt.axis('off')
plt.show()

### 2.3. Detectando e convertendo para `sv.Detections`

Chamamos `predict()` normalmente sem nenhuma menção a rastreamento e convertemos o resultado da `ultralytics` para o formato `sv.Detections`, com `sv.Detections.from_ultralytics()`. É esse formato padronizado que faz a ponte entre o detector e o rastreador da Roboflow.

In [ ]:
resultado = detect_model.predict(quadro, verbose=False)[0]
deteccoes = sv.Detections.from_ultralytics(resultado)

print(f"{len(deteccoes)} objetos detectados")

### 2.4. Rastreando com o `ByteTrackTracker`

Instanciamos o rastreador e chamamos `update(deteccoes)`, que devolve as mesmas detecções com um `tracker_id` atribuído a cada uma (`-1` para as que ainda não têm um ID confirmado).

Por padrão, o `ByteTrackTracker` só confirma um ID depois que o objeto aparece em pelo menos `minimum_consecutive_frames=2` quadros consecutivos, diferente do `ByteTrack` da `ultralytics`, que confirma o ID já no primeiro quadro. Isso reduz falsos positivos de rastreamento (uma detecção isolada e espúria não vira um ID), mas também significa que, chamando `update()` uma única vez sobre um quadro isolado, todas as detecções saem com `tracker_id` igual a `-1`. Para simular a chegada de um segundo quadro parecido com o primeiro, chamamos `update()` de novo com as mesmas detecções e alguns IDs já aparecem confirmados.

In [ ]:
rastreador = ByteTrackTracker()

deteccoes_1 = rastreador.update(deteccoes)
print("1ª chamada:", deteccoes_1.tracker_id)

deteccoes = rastreador.update(deteccoes)
print("2ª chamada:", deteccoes.tracker_id)

### 2.5. Anotando com `supervision`

Descartamos as detecções ainda sem ID confirmado (`tracker_id == -1`) e desenhamos as demais com os anotadores da `supervision`. Usando `color_lookup=sv.ColorLookup.TRACK`, cada `tracker_id` recebe uma cor própria automaticamente. No notebook anterior, precisávamos manter um dicionário `cores` à mão para conseguir o mesmo efeito.

In [ ]:
deteccoes_confirmadas = deteccoes[deteccoes.tracker_id != -1]

rotulos = [
    f"#{track_id} {resultado.names[classe_id]} {confianca:.2f}"
    for track_id, classe_id, confianca
    in zip(deteccoes_confirmadas.tracker_id, deteccoes_confirmadas.class_id, deteccoes_confirmadas.confidence)
]

anotador_caixa = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
anotador_rotulo = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.5)

quadro_anotado = anotador_caixa.annotate(quadro.copy(), deteccoes_confirmadas)
quadro_anotado = anotador_rotulo.annotate(quadro_anotado, deteccoes_confirmadas, labels=rotulos)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 3. Rastreamento no Vídeo Completo

Assim como no notebook anterior, é processando o vídeo quadro a quadro que o rastreamento mostra sua utilidade. Desta vez, vamos usar o `ByteTrackTracker` junto do `MotionEstimator`, que estima o deslocamento da própria câmera entre um quadro e outro, o que usamos para compensar esse deslocamento na hora de desenhar o rastro de cada objeto, deixando-o mais suave. Vamos entender esse componente com calma na seção 5; por enquanto, ele já entra no pipeline principal.

### 3.1. Carregando o vídeo

Assim como fizemos no notebook anterior, usamos o `cv2.VideoCapture` para abrir o vídeo e ler suas propriedades.

Diferente do notebook anterior, não precisamos recarregar o `detect_model`: como agora chamamos `predict()` em vez de `track()`, o modelo não guarda nenhum estado de rastreamento entre uma chamada e outra. Quem guarda esse estado é o `rastreador`, que instanciamos de novo a seguir.

In [ ]:
output_path = 'output/transito_rastreado_roboflow.mp4'

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_quadros = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolução: {largura}x{altura}, {fps:.1f} fps, {total_quadros} quadros")

### 3.2. Rastreador, Estimador de Movimento e Anotadores

Instanciamos tudo o que precisa manter estado entre um quadro e outro:
* `rastreador`: um `ByteTrackTracker` novo, informado do `frame_rate` real do vídeo (usado para calibrar por quantos quadros um objeto pode "sumir" antes de perder o ID).
* `estimador_movimento`: um `MotionEstimator` novo, que vai acumulando o deslocamento da câmera a partir do primeiro quadro que ele processar.
* `anotador_rastro`: um `MotionAwareTraceAnnotator`, que guarda o histórico de posições de cada `tracker_id` internamente, dispensando o dicionário `trajetorias` que mantínhamos manualmente no notebook anterior.

In [ ]:
rastreador = ByteTrackTracker(frame_rate=fps)
estimador_movimento = MotionEstimator()

anotador_caixa = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
anotador_rotulo = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.5)
anotador_rastro = MotionAwareTraceAnnotator(color_lookup=sv.ColorLookup.TRACK, trace_length=30)

### 3.3. Processando o vídeo, quadro a quadro

Para cada quadro, o laço segue sempre a mesma sequência:

1. `estimador_movimento.update(quadro)` estima o deslocamento da câmera desde o quadro anterior, devolvendo um `coord_transform`.
2. `detect_model.predict(quadro)` detecta os objetos, convertidos em seguida para `sv.Detections`.
3. `rastreador.update(deteccoes)` associa as detecções aos rastros já existentes, atribuindo o `tracker_id` de cada uma.
4. Descartamos as detecções ainda sem ID confirmado e desenhamos caixa, rótulo e rastro, passando `coord_transform` para o `anotador_rastro`, que é quem usa essa informação para compensar o movimento da câmera.

Aproveitamos o mesmo laço para contar, em `contagem`, quantos IDs únicos de cada classe passaram pelo vídeo, como no notebook anterior. Como o vídeo tem quase mil quadros em alta resolução, essa célula pode levar alguns minutos para rodar em CPU.

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (largura, altura))

contagem = defaultdict(set)

quadros_amostra = []
intervalo_amostra = max(total_quadros // 4, 1)

quadro_idx = 0
while True:
    ok, quadro = cap.read()
    if not ok:
        break

    coord_transform = estimador_movimento.update(quadro)

    resultado = detect_model.predict(quadro, verbose=False)[0]
    deteccoes = sv.Detections.from_ultralytics(resultado)
    deteccoes = rastreador.update(deteccoes)
    deteccoes = deteccoes[deteccoes.tracker_id != -1]

    for track_id, classe_id in zip(deteccoes.tracker_id, deteccoes.class_id):
        contagem[resultado.names[classe_id]].add(track_id)

    rotulos = [
        f"#{track_id} {resultado.names[classe_id]} {confianca:.2f}"
        for track_id, classe_id, confianca
        in zip(deteccoes.tracker_id, deteccoes.class_id, deteccoes.confidence)
    ]

    quadro_anotado = anotador_caixa.annotate(quadro.copy(), deteccoes)
    quadro_anotado = anotador_rastro.annotate(quadro_anotado, deteccoes, coord_transform=coord_transform)
    quadro_anotado = anotador_rotulo.annotate(quadro_anotado, deteccoes, labels=rotulos)

    writer.write(quadro_anotado)

    if quadro_idx % intervalo_amostra == 0:
        quadros_amostra.append(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))

    quadro_idx += 1

cap.release()
writer.release()

print(f"Vídeo processado salvo em '{output_path}' ({quadro_idx} quadros).")

### 3.4. Apresentando o Resultado

Assim como no notebook anterior, primeiro conferimos uma amostra dos quadros processados com o `matplotlib`, e depois tentamos exibir o vídeo completo.

In [ ]:
fig, eixos = plt.subplots(1, len(quadros_amostra), figsize=(5 * len(quadros_amostra), 5))

for eixo, quadro in zip(eixos, quadros_amostra):
    eixo.imshow(quadro)
    eixo.axis('off')

plt.tight_layout()
plt.show()

Se o seu navegador suportar o codec do vídeo gerado, ele será reproduzido logo abaixo. Caso contrário, abra o arquivo `output/transito_rastreado_roboflow.mp4` diretamente em um player de vídeo.

In [ ]:
def display_video(filename: str, **kwargs) -> None:
    """ Apresenta vídeo no notebook """
    clip = VideoFileClip(filename)
    html_embed = clip.display_in_notebook(**kwargs)
    # O moviepy cria um arquivo temporário. Só vamos apagar ele.
    Path("__temp__.mp4").unlink(missing_ok=True)
    return html_embed

In [ ]:
display_video(output_path, loop=1, width=800)

## 4. Contagem de Objetos Únicos

Como cada objeto manteve o mesmo ID do início ao fim de sua passagem pelo vídeo, o tamanho de cada conjunto em `contagem` nos dá o número de objetos únicos rastreados por classe. Bem diferente da soma de detecções por quadro, que contaria o mesmo carro dezenas de vezes.

In [ ]:
for classe, ids in sorted(contagem.items(), key=lambda item: -len(item[1])):
    print(f"{classe}: {len(ids)} objeto(s) único(s)")

## 5. O Papel do `MotionEstimator`: Rastros Mais Suaves

O `TraceAnnotator` "comum" da `supervision` desenha o rastro de um objeto ligando, em coordenadas de pixel, as posições onde ele foi visto nos últimos quadros. Isso funciona bem quando a câmera está perfeitamente parada, mas qualquer deslocamento da própria câmera (uma vibração, um zoom, uma câmera de verdade em movimento, como em um drone ou em um carro) se mistura ao movimento do objeto, distorcendo o rastro.

O `MotionEstimator` resolve isso em duas etapas, repetidas a cada quadro:

1. Escolhe pontos de referência no quadro (cantos e texturas de fundo, via detecção de *features* do OpenCV) e rastreia esses pontos entre o quadro anterior e o atual usando fluxo óptico esparso (Lucas-Kanade).
2. A partir do deslocamento desses pontos, calcula a homografia que melhor explica o movimento aparente da cena, e a acumula desde o primeiro quadro processado, criando um sistema de coordenadas "do mundo", independente da câmera.

A cada `update(quadro)`, ele devolve um `coord_transform`, capaz de converter uma posição entre as coordenadas do quadro atual e as coordenadas do mundo. Passando esse objeto para o `MotionAwareTraceAnnotator`, o rastro passa a ser desenhado nas coordenadas do mundo — cancelando o deslocamento causado pela própria câmera antes de desenhar.

Para ver esse efeito com clareza, vamos reprocessar os primeiros quadros do vídeo duas vezes: uma com o `MotionAwareTraceAnnotator` recebendo `coord_transform` (com compensação), outra com um `TraceAnnotator` comum, sem nenhuma compensação. Como cada execução precisa de um `rastreador` e um `estimador_movimento` "zerados", criamos instâncias novas a cada passagem.

In [ ]:
def processar_amostra(video_path: str, n_quadros: int, compensar_movimento: bool):
    """ Reprocessa os primeiros n_quadros do vídeo, com ou sem compensação de movimento de câmera """
    cap_amostra = cv2.VideoCapture(video_path)

    rastreador_amostra = ByteTrackTracker(frame_rate=fps)
    estimador_amostra = MotionEstimator()

    if compensar_movimento:
        anotador_rastro_amostra = MotionAwareTraceAnnotator(color_lookup=sv.ColorLookup.TRACK, trace_length=n_quadros)
    else:
        anotador_rastro_amostra = sv.TraceAnnotator(color_lookup=sv.ColorLookup.TRACK, trace_length=n_quadros)

    quadro_anotado = None
    coord_transform = None
    for _ in range(n_quadros):
        ok, quadro = cap_amostra.read()
        if not ok:
            break

        coord_transform = estimador_amostra.update(quadro)

        resultado = detect_model.predict(quadro, verbose=False)[0]
        deteccoes = sv.Detections.from_ultralytics(resultado)
        deteccoes = rastreador_amostra.update(deteccoes)
        deteccoes = deteccoes[deteccoes.tracker_id != -1]

        quadro_anotado = anotador_caixa.annotate(quadro.copy(), deteccoes)
        if compensar_movimento:
            quadro_anotado = anotador_rastro_amostra.annotate(quadro_anotado, deteccoes, coord_transform=coord_transform)
        else:
            quadro_anotado = anotador_rastro_amostra.annotate(quadro_anotado, deteccoes)

    cap_amostra.release()
    return cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB), coord_transform

n_quadros_amostra = 150  # ~5 segundos a 30 fps

quadro_sem_compensacao, _ = processar_amostra(video_path, n_quadros_amostra, compensar_movimento=False)
quadro_com_compensacao, coord_transform_final = processar_amostra(video_path, n_quadros_amostra, compensar_movimento=True)

deslocamento_x, deslocamento_y = coord_transform_final.homography_matrix[:2, 2]
print(
    f"Deslocamento de câmera acumulado em {n_quadros_amostra} quadros: "
    f"{deslocamento_x:.1f}px na horizontal, {deslocamento_y:.1f}px na vertical."
)
print("Isso mesmo com uma câmera nominalmente fixa — pequenas vibrações já bastam para desalinhar o rastro.")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(16, 7))

eixos[0].imshow(quadro_sem_compensacao)
eixos[0].set_title("Sem compensação (TraceAnnotator)")
eixos[0].axis('off')

eixos[1].imshow(quadro_com_compensacao)
eixos[1].set_title("Com compensação (MotionAwareTraceAnnotator)")
eixos[1].axis('off')

plt.tight_layout()
plt.show()

Repare como, à esquerda, os rastros ficam ligeiramente "tortos" ou deslocados em relação aos veículos (reflexo do deslocamento de câmera medido acima, que nunca é compensado). À direita, com o `MotionAwareTraceAnnotator`, os rastros seguem o objeto de forma mais reta e consistente. O efeito tende a ser sutil em uma câmera de trânsito quase fixa como esta, mas cresce muito em cenas com câmera de verdade em movimento (um drone sobrevoando uma rodovia, uma câmera de bordo em um carro, ou até um leve tremor de pan-tilt em uma câmera de segurança).

## 6. Trocando o Detector e o Rastreador: RF-DETR + OC-SORT

Na seção 1, dissemos que a filosofia BYOD permite trocar de detector sem trocar de rastreador, ou comparar algoritmos de rastreamento diferentes sobre as mesmas detecções. Vamos provar isso na prática, refazendo o pipeline da seção 3 com duas trocas:

* **Detector**: `YOLO` → `RFDETRNano`, o modelo mais leve da família RF-DETR (já usado no notebook [02_03](02_03_rf_detr.ipynb)). Ele já devolve os resultados prontos em `sv.Detections`, então não precisamos de nenhuma conversão como o `sv.Detections.from_ultralytics()`.
* **Rastreador**: `ByteTrackTracker` → `OCSORTTracker`. O OC-SORT parte do mesmo SORT (Kalman + IoU), mas ataca um problema específico: depois de uma oclusão longa, o filtro de Kalman "puro" tende a extrapolar a posição do objeto em linha reta, o que erra feio quando o objeto muda de direção. O OC-SORT corrige a trajetória estimada usando as observações reais mais recentes antes de tentar reassociar o rastro (*Observation-Centric Re-Update*), e também leva em conta a direção do movimento (`direction_consistency_weight`) na hora de decidir a que rastro uma nova detecção pertence.

O `MotionEstimator` e os anotadores continuam os mesmos conceitos da seção 3. Só precisamos de instâncias novas, já que cada uma delas guarda estado (rastros, transformação acumulada) por execução.

### 6.1. Carregando o Modelo e o Rastreador

O `RFDETRNano()` é carregado como no notebook [02_03](02_03_rf_detr.ipynb). A diferença em relação ao `predict()` da `ultralytics` é que ele espera a imagem em RGB (não BGR) e já devolve o resultado como `sv.Detections`, com a confiança mínima controlada pelo parâmetro `threshold`.

In [ ]:
detect_model_rfdetr = RFDETRNano()

rastreador_rfdetr = OCSORTTracker(frame_rate=fps)
estimador_movimento_rfdetr = MotionEstimator()

anotador_caixa_rfdetr = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
anotador_rotulo_rfdetr = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.5)
anotador_rastro_rfdetr = MotionAwareTraceAnnotator(color_lookup=sv.ColorLookup.TRACK, trace_length=30)

### 6.2. Processando o vídeo, quadro a quadro

O laço é praticamente idêntico ao da seção 3.3. As únicas mudanças são: convertemos o quadro para RGB antes de chamar `predict()`; usamos `deteccoes.data['class_name']` em vez de `resultado.names[classe_id]`, já que o RF-DETR devolve o nome da classe diretamente; e não há necessidade de nenhum `sv.Detections.from_ultralytics()`, pois `predict()` já devolve `sv.Detections`.

In [ ]:
output_path_rfdetr = 'output/transito_rastreado_rfdetr_ocsort.mp4'

cap = cv2.VideoCapture(video_path)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path_rfdetr, fourcc, fps, (largura, altura))

contagem_rfdetr = defaultdict(set)

quadros_amostra_rfdetr = []
intervalo_amostra = max(total_quadros // 4, 1)

quadro_idx = 0
while True:
    ok, quadro = cap.read()
    if not ok:
        break

    coord_transform = estimador_movimento_rfdetr.update(quadro)

    quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)
    deteccoes = detect_model_rfdetr.predict(quadro_rgb, threshold=0.5)
    deteccoes = rastreador_rfdetr.update(deteccoes)
    deteccoes = deteccoes[deteccoes.tracker_id != -1]

    for track_id, nome_classe in zip(deteccoes.tracker_id, deteccoes.data['class_name']):
        contagem_rfdetr[nome_classe].add(track_id)

    rotulos = [
        f"#{track_id} {nome_classe} {confianca:.2f}"
        for track_id, nome_classe, confianca
        in zip(deteccoes.tracker_id, deteccoes.data['class_name'], deteccoes.confidence)
    ]

    quadro_anotado = anotador_caixa_rfdetr.annotate(quadro.copy(), deteccoes)
    quadro_anotado = anotador_rastro_rfdetr.annotate(quadro_anotado, deteccoes, coord_transform=coord_transform)
    quadro_anotado = anotador_rotulo_rfdetr.annotate(quadro_anotado, deteccoes, labels=rotulos)

    writer.write(quadro_anotado)

    if quadro_idx % intervalo_amostra == 0:
        quadros_amostra_rfdetr.append(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))

    quadro_idx += 1

cap.release()
writer.release()

print(f"Vídeo processado salvo em '{output_path_rfdetr}' ({quadro_idx} quadros).")

### 6.3. Apresentando o Resultado

In [ ]:
fig, eixos = plt.subplots(1, len(quadros_amostra_rfdetr), figsize=(5 * len(quadros_amostra_rfdetr), 5))

for eixo, quadro in zip(eixos, quadros_amostra_rfdetr):
    eixo.imshow(quadro)
    eixo.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
display_video(output_path_rfdetr, loop=1, width=800)

### 6.4. Contagem de Objetos Únicos

A mesma lógica da seção 4, agora sobre `contagem_rfdetr`. Os números não precisam bater exatamente com os da seção 4: são um detector e um rastreador diferentes, com seus próprios limiares de confiança e critérios de associação, o que já é, por si só, mais uma vantagem prática da arquitetura BYOD: dá para comparar as duas combinações lado a lado sem reescrever o pipeline.

In [ ]:
for classe, ids in sorted(contagem_rfdetr.items(), key=lambda item: -len(item[1])):
    print(f"{classe}: {len(ids)} objeto(s) único(s)")